# Smart Paddy Disease Detector - Training Script

This notebook trains a MobileNetV2 model to classify paddy leaf diseases. 
Upload this to Google Colab, enable GPU connection (Runtime -> Change runtime type -> T4 GPU), and run.

In [ ]:
!pip install -q kaggle

# Note: The user requested to use a specific dataset: https://data.mendeley.com/datasets/fwcj7stb8r/1
# Please download that dataset and upload it to your Colab environment or Google Drive,
# and extract it so that 'data_dir' points to the folder containing the disease subfolders.

# If using Kaggle instead:
from google.colab import files
files.upload() # Upload kaggle.json

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# If using the Mendeley dataset you downloaded locally, you can modify this cell to unzip that dataset.
# Example for a default Kaggle dataset fallback if Mendeley is not used:
!kaggle datasets download -d vbookshelf/rice-leaf-diseases
!unzip -q rice-leaf-diseases.zip -d dataset


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import os

data_dir = 'dataset/rice_leaf_diseases' # Adjust based on dataset structure
batch_size = 32
img_size = (224, 224)

# Data Augmentation
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze base layers first

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)

In [ ]:
# Save Model
model.save('paddy_model.h5')
print("Model saved as paddy_model.h5. Download this file and put it in the root of your web app.")

from google.colab import files
files.download('paddy_model.h5')